# Hit-rate model

Thin wrapper over `ratesresearch.hitrate`: draws labelled RFQs from the declared competition model, fits the logistic, compares out-of-sample log-loss with a constant and shows the calibration. The slope is a declared band (DR-008); the fit recovers the assumption, it does not measure a market.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch import presets

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent
PRESETS = ROOT / "data" / "presets.yaml"


def show(df: pd.DataFrame) -> None:
    display(Markdown(ba.to_markdown(df)))

In [ ]:
from ratesresearch import hitrate

comp = presets.get_preset(PRESETS, "usd_rfq_competition")
x, y = hitrate.simulate_labels(comp, 40_000)
fit = hitrate.fit(x, y)
show(
    pd.DataFrame(
        {
            "feature": hitrate.FEATURES,
            "estimate": fit.beta.round(4),
            "standard_error": fit.se.round(4),
        }
    )
)
show(
    pd.DataFrame(
        [
            {
                "labels_fit": fit.n_fit,
                "labels_test": fit.n_test,
                "log_loss_model": round(fit.log_loss_model, 4),
                "log_loss_constant": round(fit.log_loss_constant, 4),
            }
        ]
    )
)

In [ ]:
rows = hitrate.calibration_table(x[fit.n_fit :], y[fit.n_fit :], fit.beta)
show(pd.DataFrame(rows, columns=["mean_predicted", "observed", "count"]).round(4))
display(SVG(hitrate.calibration_svg(rows)))